In [11]:
import cv2
import cvlib as cv
import numpy as np
import os

# Constants
MODEL_PROTO = 'deploy_gender.prototxt'
MODEL_WEIGHTS = 'gender_net.caffemodel'
GENDER_LIST = ['Male', 'Female']
INPUT_SIZE = (227, 227)
PADDING = 20
MAX_WIDTH = 800
MAX_HEIGHT = 800

# Check if model files exist
if not (os.path.exists(MODEL_PROTO) and os.path.exists(MODEL_WEIGHTS)):
    print("[ERROR] Model files not found. Please check paths.")
    exit()

# Load gender detection model
gender_model = cv2.dnn.readNetFromCaffe(MODEL_PROTO, MODEL_WEIGHTS)

# Load the image
image_path = 'images/group1.jpg'
image = cv2.imread(image_path)

if image is None:
    print(f"[ERROR] Could not load the image. Check the path: {image_path}")
    exit()

# Resize if needed (width or height too big)
(h, w) = image.shape[:2]
if w > MAX_WIDTH or h > MAX_HEIGHT:
    scale = min(MAX_WIDTH / w, MAX_HEIGHT / h)
    image = cv2.resize(image, (0, 0), fx=scale, fy=scale)
    print(f"[INFO] Image resized (new size: {image.shape[1]}x{image.shape[0]})")

# Detect faces
faces, confidences = cv.detect_face(image)

if not faces:
    print("[INFO] No faces detected!")
else:
    print(f"[INFO] Detected {len(faces)} face(s)")

    for i, face in enumerate(faces):
        startX = max(0, face[0] - PADDING)
        startY = max(0, face[1] - PADDING)
        endX = min(image.shape[1] - 1, face[2] + PADDING)
        endY = min(image.shape[0] - 1, face[3] + PADDING)

        face_crop = image[startY:endY, startX:endX]

        if face_crop.size == 0:
            continue  # Skip empty faces

        # Prepare input blob for gender model
        blob = cv2.dnn.blobFromImage(face_crop, 1.0, INPUT_SIZE,
                                     (78.4263377603, 87.7689143744, 114.895847746),
                                     swapRB=False)

        gender_model.setInput(blob)
        gender_preds = gender_model.forward()

        gender = GENDER_LIST[gender_preds[0].argmax()]
        confidence = gender_preds[0][gender_preds[0].argmax()]

        # Draw bounding box and label
        label = f"{gender}: {confidence * 100:.2f}%"
        color = (0, 255, 0) if gender == 'Male' else (255, 0, 255)  # Different color for Male/Female

        cv2.rectangle(image, (startX, startY), (endX, endY), color, 2)
        y_label = startY - 10 if startY - 10 > 10 else startY + 10
        cv2.putText(image, label, (startX, y_label), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

# Display the output
cv2.imshow("Gender Detection", image)
cv2.waitKey(0)
cv2.destroyAllWindows()



[INFO] Image resized (new size: 800x533)
[INFO] Detected 8 face(s)
